# QwerySmith v3.0 — T4 Training (Colab)

Trains the 3-seed QLoRA adapters per the pinned config. **Runtime → Change runtime type → T4 GPU.**

One environment throughout (system python + pip — no uv on Colab). Everything is config-driven: the repo carries pinned YAML per seed; this notebook only executes. Hardware, package versions, adapter hashes, and final loss are captured to `train_record.json` per seed.

**Before starting:** commit and push `datasets/olist/questions_v1.jsonl` (your reviewed question set) to the repo — Cell 1 clones it. The 9 raw CSVs come from Drive, not git.

| Cell | What it does |
|---|---|
| 1 | Setup: GPU check, deps, clone repo, raw CSVs from Drive |
| 2 | Prepare: ingest → profile → validate → retrieve → triples (CPU) |
| 3 | Train all 3 seeds (~40–60 min each on T4) |
| 4 | Contract sanity check on adapter seed 1 |
| 5 | Zip artifacts to Drive for the eval step |

In [ ]:
# ===== Cell 1: setup =====
# harness deps into system python (single env for the whole notebook)
%pip install -q typer pyyaml pydantic sqlglot sqlalchemy

import torch
assert torch.cuda.is_available(), 'Runtime -> Change runtime type -> T4 GPU'
vram = torch.cuda.get_device_properties(0).total_memory // 2**20
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {vram} MB')
assert vram >= 14000, f'need ~16GB (T4); got {vram} MB'

# repo: your fork, with the reviewed questions_v1.jsonl committed
REPO_URL = 'https://github.com/Cyrax321/QwerySmith-1.0.git'   # private? add a token: https://<token>@github.com/...
import os, subprocess
REPO = '/content/qwerysmith'
if not os.path.exists(REPO):
    subprocess.run(['git', 'clone', '-q', REPO_URL, REPO], check=True)
else:
    subprocess.run(['git', '-C', REPO, 'pull', '-q'], check=True)

from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/qwerysmith_v3'   # Kaggle CSVs live here

# link raw data into the dataset dir (the one manual step)
import shutil
os.makedirs(f'{REPO}/datasets/olist/raw', exist_ok=True)
for f in os.listdir(f'{DATA}/raw'):
    if f.endswith('.csv'):
        shutil.copy(f'{DATA}/raw/{f}', f'{REPO}/datasets/olist/raw/{f}')
n_csv = len([f for f in os.listdir(f'{REPO}/datasets/olist/raw') if f.endswith('.csv')])
print('raw files:', n_csv)
assert n_csv == 9, 'need all 9 Olist CSVs in Drive/qwerysmith_v3/raw'

# reviewed question set must be in the clone
assert os.path.exists(f'{REPO}/datasets/olist/questions_v1.jsonl'), (
    'questions_v1.jsonl not in repo — commit the reviewed set and re-run this cell')
print('setup OK')

In [ ]:
# ===== Cell 2: prepare (CPU) — ingest, profile, validate, retrieve, triples =====
import os
os.chdir(REPO)   # python -m qwery_smith resolves from the repo cwd
for stage in ['ingest', 'profile', 'validate', 'retrieve', 'triples']:
    print(f'=== {stage} ===')
    r = subprocess.run(['python', '-m', 'qwery_smith', stage, 'olist'],
                       capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print(r.stderr[-2000:])
        raise SystemExit(f'{stage} FAILED')
# validate MUST pass: scorable denominator + holdout-leak checks green

In [ ]:
# ===== Cell 3: train all 3 seeds =====
# ML deps (~3 min; unsloth resolves its own torch pin)
%pip install -q unsloth trl datasets peft transformers accelerate bitsandbytes

r = subprocess.run(['python', '-m', 'qwery_smith', 'train', 'olist',
                    '--seeds', '1,2,3', '--execute'],
                   capture_output=True, text=True)
print(r.stdout[-4000:])
if r.returncode != 0:
    print(r.stderr[-4000:])
    raise SystemExit('training FAILED')

In [ ]:
# ===== Cell 4: contract sanity check (adapter seed 1) =====
import json, yaml, pathlib
from unsloth import FastLanguageModel

run_dir = sorted(pathlib.Path(f'{REPO}/runs/olist').glob('train_*'))[-1]
cfg = yaml.safe_load(open(run_dir / 'qlora_seed1.yaml'))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=cfg['base_model'],
    adapter_name=str(run_dir / 'adapters' / 'adapter_seed1'),
    max_seq_length=cfg['batch']['max_len'],
    load_in_4bit=True,
)
FastLanguageModel.for_inference(model)

triples_path = f'{REPO}/datasets/olist/prepared/triples_seed42.jsonl'
triple = json.loads(open(triples_path).readline())
inputs = tokenizer(triple['prompt'] + '\n\nASSISTANT:\n', return_tensors='pt').to('cuda')
out = model.generate(**inputs, max_new_tokens=512, temperature=0.7, top_p=0.8, do_sample=True)
text = tokenizer.decode(out[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
print(text[:600])
assert 'SQL:' in text and ('ANSWER:' in text or 'REFUSAL:' in text), 'adapter violates output contract'
print('contract OK')

In [ ]:
# ===== Cell 5: zip artifacts to Drive =====
import pathlib, subprocess
run_dir = sorted(pathlib.Path(f'{REPO}/runs/olist').glob('train_*'))[-1]
zip_target = f'/content/drive/MyDrive/qwerysmith_v3/run_{run_dir.name}.zip'
subprocess.run(['zip', '-qr', zip_target,
                f'runs/olist/{run_dir.name}', 'datasets/olist/prepared'], cwd=REPO)
print('zipped ->', zip_target)
print('adapters inside:', [p.name for p in (run_dir / 'adapters').glob('adapter_seed*')])
print('next: eval_servers.ipynb on an L4 runtime')